# GliaSin

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.GliaSin)

class GliaSin(LinearReferenceClock):
    pass



In [3]:
model = pya.models.GliaSin()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "gliasin"
model.metadata["data_type"] = "DNA methylation"  # Paper: The model is based on DNA methylation measurements.
model.metadata["species"] = "Homo sapiens"  # Paper: The study samples are Homo sapiens.
model.metadata["year"] = 2024
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Tong, Huige, et al. \"Cell-type specific epigenetic clocks to quantify biological age at cell-type resolution.\" Aging 16 (2024): 14204–14237."
model.metadata["doi"] = "https://doi.org/10.18632/aging.206184"
model.metadata["notes"] = "Glia-Sin is a glia semi-intrinsic chronological-age clock: elastic-net regression was restricted to glia age-DMCTs but fitted to methylation values not adjusted for brain cell fractions."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["brain cortex"]  # Paper: The listed tissue is the model-development sample material.
model.metadata["predicts"] = ["chronological age"]  # Paper: The reported predictor output is chronological age.
model.metadata["training_target"] = ["chronological age"]  # Paper: The fitting outcome is chronological age.
model.metadata["unit"] = ["years"]  # Paper: The returned construct is expressed as years.
model.metadata["model_type"] = "elastic net regression"  # Paper: The clock was fitted using Elastic net.
model.metadata["platform"] = ["Illumina 450K"]  # Paper: Training/selection used Illumina 450K.
model.metadata["population"] = "adults"  # Paper: adults aged 18–97 years (Jaffe prefrontal-cortex training data)
model.metadata["journal"] = "Aging"
model.metadata["last_author"] = "Andrew E. Teschendorff"
model.metadata["n_features"] = 220
model.metadata["citations"] = 25
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
supplementary_url = "https://raw.githubusercontent.com/Duzhaozhen/OmniAge/c10fbe8cb92957520fbff1d55ae1def0691252e5/OmniAgePy/src/omniage/data/CTS/Glia-Sin.csv"
supplementary_file_name = "coefficients.csv"
os.system(f"curl -sL -o {supplementary_file_name} {supplementary_url}")

0

## Load features

In [6]:
df = pd.read_csv('coefficients.csv')
if str(df.columns[0]).startswith('Unnamed'):
    df = df.iloc[:, 1:]
mask = df['probe'].astype(str).str.lower().isin(['intercept', '(intercept)'])
intercept_value = float(df.loc[mask, 'coef'].iloc[0]) if mask.any() else 0.0
coef_df = df.loc[~mask].reset_index(drop=True)
model.features = coef_df['probe'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(coef_df['coef'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([intercept_value]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Tong, Huige, et al. "Cell-type-specific and '
             'cell-type-independent DNA methylation clocks." Aging 16 (2024).',
 'clock_name': 'gliasin',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.18632/aging.206184',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2024}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg10501210', 'cg13782884', 'cg14419975', 'cg16909962', 'cg16361302', 'cg21122225', 'cg23606718', 'cg04187185', 'cg18336674', 'cg02578944', 'cg04190475', 'cg11744777', 'cg14848772', 'cg16695576', 'cg18468088', 'cg20223728', 'cg01217984', 'cg12001630', 'cg26830108', 'cg05066959', 'cg15627457', 'cg13327545', 'cg14334161', 'cg17485681', 'cg01560972', 'c

## Normal feature ranges

Units and plausible bounds come from `pyaging`'s feature range registry, keyed by feature name with a fallback to the default for the clock's `data_type`. `predict_age` warns when input values fall outside these bounds, which usually means the data is in different units than the clock expects.

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: coefficients.csv
